<a href="https://colab.research.google.com/github/Juno-Wong/Dome-41/blob/main/Dome_41.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Domesphere AI-powered Dashboard**

In [3]:
import pandas as pd

# Load all sheets
all_sheets = pd.read_excel("shameless_dashboard_200users.xlsx", sheet_name=None)

# Loop through each sheet and save as CSV
for sheet_name, df in all_sheets.items():
    df.to_csv(f"{sheet_name}.csv", index=False)

# **Clean the dataset**

## **1. Understand the structure**

In [7]:
df_users = pd.read_csv("shameless_users.csv")
df_posts = pd.read_csv("shameless_posts.csv")
df_comments = pd.read_csv("shameless_comments.csv")

In [9]:
df_users.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   user_id     200 non-null    object
 1   created_at  200 non-null    object
 2   username    200 non-null    object
 3   city        200 non-null    object
 4   industry    200 non-null    object
 5   gender      200 non-null    object
dtypes: object(6)
memory usage: 9.5+ KB


In [11]:
df_posts.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84 entries, 0 to 83
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   post_id              84 non-null     object
 1   user_id              84 non-null     object
 2   post_username        84 non-null     object
 3   post_created_at      84 non-null     object
 4   heading              84 non-null     object
 5   content              84 non-null     object
 6   embed_platform       44 non-null     object
 7   post_like_count      84 non-null     int64 
 8   post_like_usernames  66 non-null     object
 9   comment_count        84 non-null     int64 
 10  latest_comment_at    70 non-null     object
dtypes: int64(2), object(9)
memory usage: 7.3+ KB


In [13]:
df_comments.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 191 entries, 0 to 190
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   comment_id           191 non-null    object
 1   post_id              191 non-null    object
 2   user_id              191 non-null    object
 3   comment_username     191 non-null    object
 4   comment_created_at   191 non-null    object
 5   content              191 non-null    object
 6   reply_to_comment_id  30 non-null     object
 7   comment_like_count   191 non-null    int64 
dtypes: int64(1), object(7)
memory usage: 12.1+ KB


## **2. Fix Data Types**

Ensures temporal consistency and enables downstream time-based analysis

In [17]:
# Users
df_users["created_at"] = pd.to_datetime(df_users["created_at"])

In [19]:
# Posts
df_posts["post_created_at"] = pd.to_datetime(df_posts["post_created_at"])
df_posts["latest_comment_at"] = pd.to_datetime(df_posts["latest_comment_at"])

In [21]:
# Comments
df_comments["comment_created_at"] = pd.to_datetime(df_comments["comment_created_at"])

### **3. Handle Missing Values (SMART handling only)**

**3.1 Posts**

In [25]:
df_posts["embed_platform"] = df_posts["embed_platform"].fillna("None")

In [27]:
df_posts["post_like_usernames"] = df_posts["post_like_usernames"].fillna("")

In [29]:
df_posts["latest_comment_at"] = df_posts["latest_comment_at"].fillna(df_posts["post_created_at"])

**3.2 Comments**

In [32]:
# Keep as is
# reply_to_comment_id stays NaN

Null values represent top-level comments (not replies).

*Explain: The reply_to_comment_id field contains null values for top-level comments, which do not reply to any existing comment. These null values were preserved, as they represent meaningful structural information rather than missing or erroneous data.*

### **4. Standardise Column Names**

In [36]:
df_users.rename(columns={"created_at": "user_created_at"})

,user_id,user_created_at,username,city,industry,gender
0,92a2012e-738d-4a74-becf-07dd07ab44f7,2026-01-07 08:36:00+00:00,lila,Wellington,Advertising,female
1,ec455a41-4c72-4c7d-9ace-89f42929f39e,2026-01-07 14:12:00+00:00,brookea,London,Student,female
2,03ec60cf-e03a-4715-87b7-3a25540371c5,2026-01-09 19:18:00+00:00,penny66,Bristol,Education,female
3,51efd9cc-9dd5-4483-b6de-ee8e728b82e9,2026-01-10 21:31:00+00:00,paige69,Brisbane,Healthcare,female
4,3c46fc4c-22a4-4dca-9078-1eb0d5d529cc,2026-01-12 09:48:00+00:00,mads,London,Legal,male
...,...,...,...,...,...,...
195,87f6beaf-55e7-47ea-80f0-70cda16730d2,2026-04-01 17:44:00+00:00,alixzz,Melbourne,Student,female
196,ec558623-0f9d-4d8b-88b5-51e03cea6aa4,2026-04-01 17:44:00+00:00,elsieclub,Auckland,Technology,female
197,a4dbd257-6e33-4ccb-bd1e-7ab23ad6c4c8,2026-04-02 21:18:00+00:00,jessie,Singapore,Education,female
198,27ce0655-01cf-4fa8-81e3-021f45b37a31,2026-04-04 07:37:00+00:00,bonnie_x,Christchurch,Advertising,female


Timestamp fields were preserved with distinct naming (e.g., `post_created_at`, `comment_created_at`, `user_created_at`) to maintain semantic clarity, as they represent different events within the system. While schema standardisation was considered, preserving meaningful distinctions was prioritised to support accurate downstream analysis.

### **5. Ensure Key Consistency**

Check relationships -> Remove orphan records and ensure relational integrity.

In [41]:
# Posts must have valid users
df_posts = df_posts[df_posts["user_id"].isin(df_users["user_id"])]

In [43]:
# Comments must have valid posts
df_comments = df_comments[df_comments["post_id"].isin(df_posts["post_id"])]

*Explain: Referential integrity was enforced by ensuring that all foreign key relationships were valid. Specifically, posts were filtered to include only records with `user_id` values present in the users dataset, and comments were filtered to include only valid `post_id` references. This prevents orphan records and ensures consistency across datasets.*

### **6. Remove Duplicates**

In [47]:
df_users = df_users.drop_duplicates()

In [49]:
df_posts = df_posts.drop_duplicates()

In [51]:
df_comments = df_comments.drop_duplicates()

Duplicate records were removed from all datasets to prevent redundancy and ensure data accuracy. This step helps avoid issues such as double counting and inconsistent analysis in downstream processes.

### **7. Clean Text**

In [55]:
import re

def basic_clean(text):
    text = str(text).strip()
    return text

df_posts["content"] = df_posts["content"].apply(basic_clean)
df_comments["content"] = df_comments["content"].apply(basic_clean)

Text fields were standardised by converting all values to string format and removing leading and trailing whitespace. This ensures consistency in textual data and prevents formatting-related issues in downstream processing.

### **8. Sort Data**

In [59]:
df_posts = df_posts.sort_values(by="post_created_at")
df_comments = df_comments.sort_values(by="comment_created_at")

###**9. Cleaned Output**

In [61]:
df_users.to_csv("clean_users.csv", index=False)
df_posts.to_csv("clean_posts.csv", index=False)
df_comments.to_csv("clean_comments.csv", index=False)

# **Feature Engineering**

In [63]:
import numpy as np
import re
from collections import Counter, defaultdict

from sklearn.feature_extraction.text import TfidfVectorizer

###Load Cleaned Data

In [65]:
users = pd.read_csv("clean_users.csv")
posts = pd.read_csv("clean_posts.csv")
comments = pd.read_csv("clean_comments.csv")

#rename columns
if "created_at" in users.columns and "user_created_at" not in users.columns:
    users = users.rename(columns={"created_at": "user_created_at"})

###Convert date columns into datetime

In [67]:
users["user_created_at"] = pd.to_datetime(users["user_created_at"], errors="coerce")
posts["post_created_at"] = pd.to_datetime(posts["post_created_at"], errors="coerce")
posts["latest_comment_at"] = pd.to_datetime(posts["latest_comment_at"], errors="coerce")
comments["comment_created_at"] = pd.to_datetime(comments["comment_created_at"], errors="coerce")

###Assisting Functions - Clean text, Sentiment labels, Saving outputs and Clean text columns in posts and comments

In [69]:
def clean_text(text):
    """Basic text normalisation for NLP features."""
    text = str(text).lower().strip()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text

def sentiment_label(score):
    if score >= 0.05:
        return "positive"
    elif score <= -0.05:
        return "negative"
    return "neutral"

def save_feature(df, filename):
    df.to_csv(filename, index=False)
    print(f"Saved: {filename}")

posts["heading_clean"] = posts["heading"].fillna("").apply(clean_text)
posts["content_clean"] = posts["content"].fillna("").apply(clean_text)
posts["full_text"] = (posts["heading_clean"] + " " + posts["content_clean"]).str.strip()

comments["content_clean"] = comments["content"].fillna("").apply(clean_text)

###Building Event Tables

In [71]:
# Posts as events
post_events = posts[["user_id", "post_created_at"]].copy()
post_events["event_type"] = "post"
post_events = post_events.rename(columns={"post_created_at": "event_time"})

# Comments as events
comment_events = comments[["user_id", "comment_created_at"]].copy()
comment_events["event_type"] = "comment"
comment_events = comment_events.rename(columns={"comment_created_at": "event_time"})

# Combined interaction events
events = pd.concat([post_events, comment_events], ignore_index=True)
events["event_date"] = events["event_time"].dt.date
events["event_hour"] = events["event_time"].dt.hour
events["event_week"] = events["event_time"].dt.to_period("W").astype(str)
events["event_month"] = events["event_time"].dt.to_period("M").astype(str)

C:\Users\User\AppData\Local\Temp\ipykernel_14892\2787294732.py:15: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  events["event_week"] = events["event_time"].dt.to_period("W").astype(str)
C:\Users\User\AppData\Local\Temp\ipykernel_14892\2787294732.py:16: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  events["event_month"] = events["event_time"].dt.to_period("M").astype(str)


###Community Features

####Feature 1: Active Listeners Over Time

In [74]:
# Count distinct users per day from interaction records
active_listeners_daily = (
    events.groupby("event_date")["user_id"]
    .nunique()
    .reset_index(name="active_listeners")
)

active_listener_kpi = pd.DataFrame({
    "metric": ["total_active_days", "avg_daily_active_listeners", "max_daily_active_listeners"],
    "value": [
        active_listeners_daily["event_date"].nunique(),
        round(active_listeners_daily["active_listeners"].mean(), 2),
        active_listeners_daily["active_listeners"].max()
    ]
})

save_feature(active_listeners_daily, "feature_01_active_listeners_over_time.csv")
save_feature(active_listener_kpi, "feature_01_active_listeners_kpi.csv")

print("\n=== Feature 1: Active Listeners Over Time ===")
print(active_listeners_daily.head(10))
print("\n=== Feature 1 KPI ===")
print(active_listener_kpi)

Saved: feature_01_active_listeners_over_time.csv
Saved: feature_01_active_listeners_kpi.csv

=== Feature 1: Active Listeners Over Time ===
   event_date  active_listeners
0  2026-01-09                 2
1  2026-01-14                 2
2  2026-01-15                 2
3  2026-01-16                 2
4  2026-01-17                 4
5  2026-01-25                 5
6  2026-01-26                 4
7  2026-01-27                 1
8  2026-01-28                 3
9  2026-01-29                 2

=== Feature 1 KPI ===
                       metric  value
0           total_active_days  64.00
1  avg_daily_active_listeners   4.03
2  max_daily_active_listeners  11.00


####Feature 2: Audience Segmentation

In [76]:
# Count posts, comments, and received likes by city/gender/industry
post_counts_user = posts.groupby("user_id").size().reset_index(name="post_count")
comment_counts_user = comments.groupby("user_id").size().reset_index(name="comment_count")

# We only know like counts received, not who gave the likes.
post_likes_received = posts.groupby("user_id")["post_like_count"].sum().reset_index(name="post_likes_received")
comment_likes_received = comments.groupby("user_id")["comment_like_count"].sum().reset_index(name="comment_likes_received")

user_engagement = users[["user_id", "username", "city", "industry", "gender"]].copy()
for df in [post_counts_user, comment_counts_user, post_likes_received, comment_likes_received]:
    user_engagement = user_engagement.merge(df, on="user_id", how="left")

for col in ["post_count", "comment_count", "post_likes_received", "comment_likes_received"]:
    user_engagement[col] = user_engagement[col].fillna(0)

user_engagement["total_interactions"] = (
    user_engagement["post_count"] +
    user_engagement["comment_count"] +
    user_engagement["post_likes_received"] +
    user_engagement["comment_likes_received"]
)

audience_segment_city = (
    user_engagement.groupby("city")[["post_count", "comment_count", "post_likes_received", "comment_likes_received", "total_interactions"]]
    .sum()
    .reset_index()
    .sort_values("total_interactions", ascending=False)
)

audience_segment_gender = (
    user_engagement.groupby("gender")[["post_count", "comment_count", "post_likes_received", "comment_likes_received", "total_interactions"]]
    .sum()
    .reset_index()
    .sort_values("total_interactions", ascending=False)
)

audience_segment_industry = (
    user_engagement.groupby("industry")[["post_count", "comment_count", "post_likes_received", "comment_likes_received", "total_interactions"]]
    .sum()
    .reset_index()
    .sort_values("total_interactions", ascending=False)
)

save_feature(audience_segment_city, "feature_02_audience_segmentation_city.csv")
save_feature(audience_segment_gender, "feature_02_audience_segmentation_gender.csv")
save_feature(audience_segment_industry, "feature_02_audience_segmentation_industry.csv")

print("\n=== Feature 2: Audience Segmentation by City ===")
print(audience_segment_city.head(10))
print("\n=== Feature 2: Audience Segmentation by Gender ===")
print(audience_segment_gender)
print("\n=== Feature 2: Audience Segmentation by Industry ===")
print(audience_segment_industry.head(10))

Saved: feature_02_audience_segmentation_city.csv
Saved: feature_02_audience_segmentation_gender.csv
Saved: feature_02_audience_segmentation_industry.csv

=== Feature 2: Audience Segmentation by City ===
            city  post_count  comment_count  post_likes_received  \
8         London        16.0           24.0                 35.0   
15    Wellington         9.0           26.0                 14.0   
3        Bristol         9.0           24.0                 11.0   
1       Auckland        10.0           16.0                 18.0   
9     Manchester         6.0           13.0                  9.0   
6     Gold Coast         8.0           11.0                 17.0   
2       Brisbane         6.0           13.0                 13.0   
10     Melbourne         8.0           12.0                 11.0   
5   Christchurch         5.0            8.0                 13.0   
0       Adelaide         4.0           11.0                  4.0   

    comment_likes_received  total_interactions  

####Feature 3: Peak Activity Time

In [78]:
#Count interactions by hour using posts + comments
peak_activity_hour = (
    events.groupby("event_hour")
    .size()
    .reset_index(name="interaction_count")
    .sort_values("event_hour")
)

peak_hour = peak_activity_hour.loc[peak_activity_hour["interaction_count"].idxmax(), "event_hour"]
peak_hour_kpi = pd.DataFrame({
    "metric": ["peak_hour", "peak_hour_interactions"],
    "value": [int(peak_hour), int(peak_activity_hour["interaction_count"].max())]
})

save_feature(peak_activity_hour, "feature_03_peak_activity_time.csv")
save_feature(peak_hour_kpi, "feature_03_peak_activity_kpi.csv")

print("\n=== Feature 3: Peak Activity Time ===")
print(peak_activity_hour)
print("\n=== Feature 3 KPI ===")
print(peak_hour_kpi)

Saved: feature_03_peak_activity_time.csv
Saved: feature_03_peak_activity_kpi.csv

=== Feature 3: Peak Activity Time ===
    event_hour  interaction_count
0            0                  9
1            1                 11
2            2                  8
3            3                  7
4            4                  5
5            5                  7
6            6                  8
7            7                 12
8            8                 19
9            9                 21
10          10                 12
11          11                 24
12          12                 18
13          13                 15
14          14                 10
15          15                  7
16          16                  5
17          17                 11
18          18                 15
19          19                 12
20          20                 14
21          21                 10
22          22                  5
23          23                 10

=== Feature 3 KPI ===
       

####Feature 4: Top Contributors

In [80]:
# Count posts + comments + likes received per user
top_contributors = user_engagement[
    ["user_id", "username", "city", "industry", "gender",
     "post_count", "comment_count", "post_likes_received", "comment_likes_received", "total_interactions"]
].sort_values("total_interactions", ascending=False)

save_feature(top_contributors, "feature_04_top_contributors.csv")

print("\n=== Feature 4: Top Contributors ===")
print(top_contributors.head(20))

Saved: feature_04_top_contributors.csv

=== Feature 4: Top Contributors ===
                                 user_id    username          city  \
4   3c46fc4c-22a4-4dca-9078-1eb0d5d529cc        mads        London   
0   92a2012e-738d-4a74-becf-07dd07ab44f7        lila    Wellington   
72  1fd42c2d-fadb-4794-8d33-f4eefae244ac        cleo      Auckland   
8   8c3853ef-e9fd-489e-a00e-c4e6de6388c3        soph       Bristol   
92  f97fc0e9-dace-4428-8859-ad8ebd2532fd        tara     Newcastle   
15  7b71142d-776d-422c-9dd8-e8a1db4563e1       layla    Wellington   
28  c46eb7b5-1cd6-4b09-97c3-0cb70933c953        fern    Manchester   
91  0bf84a80-6729-4c12-80af-652ef37f5f0b        beth  Christchurch   
3   51efd9cc-9dd5-4483-b6de-ee8e728b82e9     paige69      Brisbane   
5   b3cbce25-bcfc-4039-8e81-460817f8514c  poppy_club        London   
1   ec455a41-4c72-4c7d-9ace-89f42929f39e     brookea        London   
30  971b74b9-34b6-4d68-b86b-6374e2cf764f       remis       Bristol   
94  5ba27d3d-a

###Content Features

In [90]:
#Topic Extraction Function
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=100,
    ngram_range=(1, 2),
    min_df=1
)

post_tfidf = vectorizer.fit_transform(posts["full_text"])
feature_names = np.array(vectorizer.get_feature_names_out())

# Dominant topic for each post
top_term_idx = np.asarray(post_tfidf.argmax(axis=1)).ravel()
posts["dominant_topic"] = feature_names[top_term_idx]

####Feature 5: Trending Topics

In [95]:
# Most discussed keywords/topics in posts and comments

combined_text = posts["full_text"].fillna("")

custom_stopwords = [
    "think", "people", "just", "want", "really", "actually", "interesting",
    "love", "best", "good", "great", "dont", "didnt", "doesnt", "feel",
    "feels", "going", "doing", "coming", "need", "read", "know", "like",
    "episode", "podcast", "online", "fully", "angle", "pr"
]

topic_vectorizer = TfidfVectorizer(
    stop_words=custom_stopwords,
    max_features=50,
    ngram_range=(2, 8),   # focus on recurring phrases instead of single words
    min_df=2
)

topic_matrix = topic_vectorizer.fit_transform(combined_text)
topic_scores = np.asarray(topic_matrix.sum(axis=0)).ravel()
topic_terms = topic_vectorizer.get_feature_names_out()

trending_topics = pd.DataFrame({
    "topic_phrase": topic_terms,
    "score": topic_scores
}).sort_values("score", ascending=False).reset_index(drop=True)

save_feature(trending_topics, "feature_05_trending_topics.csv")

print("\n=== Feature 5: Recurring Topics / Phrases ===")
print(trending_topics.head(20))

Saved: feature_05_trending_topics.csv

=== Feature 5: Recurring Topics / Phrases ===
                                        topic_phrase      score
0                                           than the  20.410225
1                                            what is   8.044237
2                                         would your   4.022119
3                            trying to separate what   4.022119
4                                 trying to separate   4.022119
5                                          trying to   4.022119
6                 separate what is from what is loud   4.022119
7                      separate what is from what is   4.022119
8                         separate what is from what   4.022119
9                              separate what is from   4.022119
10                                  separate what is   4.022119
11                                     separate what   4.022119
12  curious whether changed their mind over the last   3.586284
13      whether cha

####Feature 6: Trending Conversations

In [97]:
# Rank posts by comments + likes
trending_conversations = posts[
    ["post_id", "post_username", "post_created_at", "heading", "post_like_count", "comment_count", "dominant_topic"]
].copy()

trending_conversations["conversation_score"] = (
    trending_conversations["post_like_count"] + trending_conversations["comment_count"]
)

trending_conversations = trending_conversations.sort_values(
    "conversation_score", ascending=False
).reset_index(drop=True)

save_feature(trending_conversations, "feature_06_trending_conversations.csv")

print("\n=== Feature 6: Trending Conversations ===")
print(trending_conversations.head(20))

Saved: feature_06_trending_conversations.csv

=== Feature 6: Trending Conversations ===
                                 post_id post_username  \
0   e1d99ba5-08bb-4cb6-86b0-9b5aaae3ee28          beth   
1   c09b7467-d5f1-456c-8f6e-5d015f0248b7    poppy_club   
2   a5805a92-36f4-4cc4-b616-30b62410a9f6       josie75   
3   96893c7b-7acc-4d2e-820f-a2977f4e8af8        nell35   
4   60165a64-6147-4751-a15a-89cc89847ccf         elsie   
5   59a7b8cf-33d1-4d84-855f-e02b9045476f           mae   
6   f5f35d98-c72f-43fe-aac3-c33bde9ce310          mads   
7   d9bb1bd3-1106-4db0-9c63-2968dd55af46       hallezz   
8   45917656-6d24-4051-b55f-c6590833f526          beth   
9   c9f7127c-9264-451c-b158-7b0ccf7aff0b          mads   
10  4ce780f8-fac3-49d1-b04a-6157c0daaac6    poppy_club   
11  1b3e2721-f5d7-434b-a363-69c6a7f2a42c        isla_x   
12  0e3815a8-14f2-4da9-bac0-254a9b9149de          lila   
13  e1edcf7b-0b8b-40b8-b643-d60f9af9051e          lila   
14  86e188be-c8ac-4809-8637-6528b92f210b  

####Feature 7: Sentiment Analysis

In [99]:
#VADER - sentiment analysis tool (Positive, Negative, Neutral)
try:
    from nltk.sentiment import SentimentIntensityAnalyzer
    import nltk

    try:
        nltk.data.find("sentiment/vader_lexicon.zip")
    except LookupError:
        nltk.download("vader_lexicon")

    USE_VADER = True
except:
    USE_VADER = False

# Apply sentiment scoring
if USE_VADER:
    sia = SentimentIntensityAnalyzer()

    posts["sentiment_score"] = posts["full_text"].apply(
        lambda x: sia.polarity_scores(str(x))["compound"]
    )
    comments["sentiment_score"] = comments["content_clean"].apply(
        lambda x: sia.polarity_scores(str(x))["compound"]
    )

else:
    positive_words = {
        "love", "great", "good", "amazing", "fun", "helpful",
        "interesting", "best", "excited", "happy"
    }
    negative_words = {
        "bad", "boring", "hate", "annoying", "worst", "confusing",
        "poor", "hard", "sad", "frustrating"
    }

    def simple_sentiment(text):
        words = set(str(text).lower().split())
        pos = len(words & positive_words)
        neg = len(words & negative_words)

        if pos + neg == 0:
            return 0.0
        return (pos - neg) / (pos + neg)

    posts["sentiment_score"] = posts["full_text"].apply(simple_sentiment)
    comments["sentiment_score"] = comments["content_clean"].apply(simple_sentiment)

# Convert score into label
def sentiment_label(score):
    if score >= 0.05:
        return "positive"
    elif score <= -0.05:
        return "negative"
    return "neutral"

posts["sentiment_label"] = posts["sentiment_score"].apply(sentiment_label)
comments["sentiment_label"] = comments["sentiment_score"].apply(sentiment_label)

# Summaries
sentiment_summary_posts = (
    posts.groupby("sentiment_label")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

sentiment_summary_comments = (
    comments.groupby("sentiment_label")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

save_feature(sentiment_summary_posts, "feature_07_sentiment_posts.csv")
save_feature(sentiment_summary_comments, "feature_07_sentiment_comments.csv")

print("\n=== Feature 7: Sentiment Summary (Posts) ===")
print(sentiment_summary_posts)

print("\n=== Feature 7: Sentiment Summary (Comments) ===")
print(sentiment_summary_comments)

Saved: feature_07_sentiment_posts.csv
Saved: feature_07_sentiment_comments.csv

=== Feature 7: Sentiment Summary (Posts) ===
  sentiment_label  count
1        positive     76
0        negative      8

=== Feature 7: Sentiment Summary (Comments) ===
  sentiment_label  count
2        positive     94
1         neutral     63
0        negative     34


####Feature 8: Content Performance by Topic

In [101]:
# Group posts by dominant topic
content_performance_by_topic = (
    posts.groupby("dominant_topic")
    .agg(
        post_count=("post_id", "count"),
        avg_post_likes=("post_like_count", "mean"),
        avg_comments=("comment_count", "mean"),
        total_post_likes=("post_like_count", "sum"),
        total_comments=("comment_count", "sum")
    )
    .reset_index()
)

content_performance_by_topic["engagement_score"] = (
    content_performance_by_topic["avg_post_likes"] +
    content_performance_by_topic["avg_comments"]
)

content_performance_by_topic = content_performance_by_topic.sort_values(
    "engagement_score", ascending=False
)

save_feature(content_performance_by_topic, "feature_08_content_performance_by_topic.csv")

print("\n=== Feature 8: Content Performance by Topic ===")
print(content_performance_by_topic.head(20))


Saved: feature_08_content_performance_by_topic.csv

=== Feature 8: Content Performance by Topic ===
          dominant_topic  post_count  avg_post_likes  avg_comments  \
7                changed           1        3.000000      3.000000   
18                  read           2        2.500000      2.500000   
12                  dumb           9        2.444444      2.555556   
16                  like           1        3.000000      2.000000   
8               circling           3        1.000000      3.666667   
11             discourse           8        2.000000      2.625000   
15                 group           7        2.285714      2.285714   
19              smartest           2        3.500000      1.000000   
2                  angle           6        1.833333      2.333333   
5          best substack           6        1.666667      2.500000   
17                  open          11        1.818182      2.272727   
9                  click           1        1.000000      3.

###Commercial Features

####Feature 9: Engagement Rate

In [104]:
posts_daily = posts.groupby(posts["post_created_at"].dt.date).size().reset_index(name="posts")
comments_daily = comments.groupby(comments["comment_created_at"].dt.date).size().reset_index(name="comments")
post_likes_daily = posts.groupby(posts["post_created_at"].dt.date)["post_like_count"].sum().reset_index(name="post_likes")
comment_likes_daily = comments.groupby(comments["comment_created_at"].dt.date)["comment_like_count"].sum().reset_index(name="comment_likes")

engagement_rate = posts_daily.merge(
    comments_daily,
    left_on="post_created_at",
    right_on="comment_created_at",
    how="outer"
)

engagement_rate = engagement_rate.rename(columns={"post_created_at": "date"}).drop(
    columns=["comment_created_at"], errors="ignore"
)

engagement_rate = engagement_rate.merge(
    post_likes_daily.rename(columns={"post_created_at": "date"}),
    on="date",
    how="outer"
)

engagement_rate = engagement_rate.merge(
    comment_likes_daily.rename(columns={"comment_created_at": "date"}),
    on="date",
    how="outer"
)

# Fill only numeric columns, not the date column
for col in ["posts", "comments", "post_likes", "comment_likes"]:
    engagement_rate[col] = engagement_rate[col].fillna(0)

engagement_rate = engagement_rate.sort_values("date")

engagement_rate["total_interactions"] = (
    engagement_rate["posts"] +
    engagement_rate["comments"] +
    engagement_rate["post_likes"] +
    engagement_rate["comment_likes"]
)

total_members = users["user_id"].nunique()
engagement_rate["total_members"] = total_members
engagement_rate["engagement_rate_pct"] = (
    engagement_rate["total_interactions"] / engagement_rate["total_members"]
) * 100

save_feature(engagement_rate, "feature_09_engagement_rate.csv")

print("\n=== Feature 9: Engagement Rate ===")
print(engagement_rate.head(20))

Saved: feature_09_engagement_rate.csv

=== Feature 9: Engagement Rate ===
          date  posts  comments  post_likes  comment_likes  \
0   2026-01-09    1.0       1.0         2.0            3.0   
1   2026-01-14    1.0       1.0         4.0            1.0   
2   2026-01-15    0.0       0.0         0.0            3.0   
3   2026-01-16    1.0       1.0         3.0            3.0   
4   2026-01-17    1.0       5.0         3.0            2.0   
5   2026-01-25    2.0       3.0         3.0            3.0   
6   2026-01-26    1.0       4.0         4.0            2.0   
7   2026-01-27    1.0       0.0         2.0            0.0   
8   2026-01-28    2.0       2.0         1.0            1.0   
9   2026-01-29    0.0       0.0         0.0            4.0   
10  2026-01-30    0.0       0.0         0.0            2.0   
11  2026-01-31    3.0       1.0         6.0            1.0   
12  2026-02-01    1.0       5.0         2.0            3.0   
13  2026-02-02    1.0       2.0         4.0            4.0

####Feature 10: Retention Rate

In [106]:
# Returning users that are active in consecutive weeks

# Create week start from event timestamps
events["week_start"] = events["event_time"].dt.to_period("W").apply(lambda r: r.start_time)

# Weekly user activity
weekly_user_activity = (
    events.groupby(["week_start", "user_id"])
    .size()
    .reset_index(name="activity_count")
    .sort_values(["user_id", "week_start"])
)

# Previous active week for each user
weekly_user_activity["prev_week"] = weekly_user_activity.groupby("user_id")["week_start"].shift(1)

# Mark returning users
weekly_user_activity["returned"] = (
    (weekly_user_activity["prev_week"].notna()) &
    ((weekly_user_activity["week_start"] - weekly_user_activity["prev_week"]).dt.days <= 7)
)

# Retention summary
retention_rate = (
    weekly_user_activity.groupby("week_start")
    .agg(
        active_users=("user_id", "nunique"),
        returning_users=("returned", "sum")
    )
    .reset_index()
)

retention_rate["retention_rate_pct"] = np.where(
    retention_rate["active_users"] > 0,
    (retention_rate["returning_users"] / retention_rate["active_users"]) * 100,
    0
)

# Merge usernames so active users can be shown by name
weekly_active_users = weekly_user_activity.merge(
    users[["user_id", "username"]],
    on="user_id",
    how="left"
)

# Create a list of active usernames for each week
active_user_names_by_week = (
    weekly_active_users.groupby("week_start")["username"]
    .apply(lambda names: ", ".join(sorted(names.dropna().unique())))
    .reset_index(name="active_user_names")
)

# Combine names into main retention table
retention_rate = retention_rate.merge(active_user_names_by_week, on="week_start", how="left")
retention_rate = retention_rate.merge(returning_user_names_by_week, on="week_start", how="left")

# Fill empty returning user names
retention_rate["returning_user_names"] = retention_rate["returning_user_names"].fillna("")

save_feature(retention_rate, "feature_10_retention_rate.csv")
save_feature(active_user_names_by_week, "feature_10_active_user_names_by_week.csv")

print("\n=== Feature 10: Retention Rate ===")
print(retention_rate)

print("\n=== Feature 10: Active User Names by Week ===")
print(active_user_names_by_week)

C:\Users\User\AppData\Local\Temp\ipykernel_14892\1278462475.py:4: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  events["week_start"] = events["event_time"].dt.to_period("W").apply(lambda r: r.start_time)


NameError: name 'returning_user_names_by_week' is not defined

####Feature 11: Audience Value Segments

In [111]:
# Segmenting audience by engagement score + demographics
audience_value = user_engagement.copy()

# Weighted score:
# posts = 3, comments = 2, likes received = 1
audience_value["engagement_score"] = (
    audience_value["post_count"] * 3 +
    audience_value["comment_count"] * 2 +
    audience_value["post_likes_received"] * 1 +
    audience_value["comment_likes_received"] * 1
)

# Segment by score quantiles
audience_value["value_segment"] = pd.qcut(
    audience_value["engagement_score"].rank(method="first"),
    q=3,
    labels=["low", "medium", "high"]
)

audience_value_segments = audience_value[
    ["user_id", "username", "city", "industry", "gender",
     "post_count", "comment_count", "post_likes_received", "comment_likes_received",
     "engagement_score", "value_segment"]
].sort_values("engagement_score", ascending=False)

audience_value_segment_summary = (
    audience_value_segments.groupby(["value_segment", "city", "industry", "gender"])
    .size()
    .reset_index(name="user_count")
    .sort_values(["value_segment", "user_count"], ascending=[True, False])
)

save_feature(audience_value_segments, "feature_11_audience_value_segments.csv")
save_feature(audience_value_segment_summary, "feature_11_audience_value_segment_summary.csv")

print("\n=== Feature 11: Audience Value Segments ===")
print(audience_value_segments.head(20))
print("\n=== Feature 11: Audience Value Segment Summary ===")
print(audience_value_segment_summary.head(20))

Saved: feature_11_audience_value_segments.csv
Saved: feature_11_audience_value_segment_summary.csv

=== Feature 11: Audience Value Segments ===
                                 user_id    username          city  \
4   3c46fc4c-22a4-4dca-9078-1eb0d5d529cc        mads        London   
0   92a2012e-738d-4a74-becf-07dd07ab44f7        lila    Wellington   
72  1fd42c2d-fadb-4794-8d33-f4eefae244ac        cleo      Auckland   
8   8c3853ef-e9fd-489e-a00e-c4e6de6388c3        soph       Bristol   
92  f97fc0e9-dace-4428-8859-ad8ebd2532fd        tara     Newcastle   
28  c46eb7b5-1cd6-4b09-97c3-0cb70933c953        fern    Manchester   
15  7b71142d-776d-422c-9dd8-e8a1db4563e1       layla    Wellington   
91  0bf84a80-6729-4c12-80af-652ef37f5f0b        beth  Christchurch   
3   51efd9cc-9dd5-4483-b6de-ee8e728b82e9     paige69      Brisbane   
5   b3cbce25-bcfc-4039-8e81-460817f8514c  poppy_club        London   
30  971b74b9-34b6-4d68-b86b-6374e2cf764f       remis       Bristol   
1   ec455a41-4c7

C:\Users\User\AppData\Local\Temp\ipykernel_14892\4265582373.py:27: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  audience_value_segments.groupby(["value_segment", "city", "industry", "gender"])


####Feature 12: Sponsor Opportunity Insights

In [113]:
# Combine trending topics + high-value audience segments to provide recommendations

# High-value audience demographics
high_value_users = audience_value_segments[audience_value_segments["value_segment"] == "high"]

high_value_demographics = (
    high_value_users.groupby(["city", "industry", "gender"])
    .size()
    .reset_index(name="high_value_user_count")
    .sort_values("high_value_user_count", ascending=False)
)

# Topic by demographic evidence
posts_with_users = posts.merge(
    users[["user_id", "city", "industry", "gender"]],
    on="user_id",
    how="left"
)

topic_segment_evidence = (
    posts_with_users.groupby(["dominant_topic", "city", "industry", "gender"])
    .agg(
        post_count=("post_id", "count"),
        total_likes=("post_like_count", "sum"),
        total_comments=("comment_count", "sum")
    )
    .reset_index()
)

topic_segment_evidence["engagement_score"] = (
    topic_segment_evidence["post_count"] +
    topic_segment_evidence["total_likes"] +
    topic_segment_evidence["total_comments"]
)

# Remove weak or irrelevant topics
bad_topics = [
    "than the", "for the", "what is", "mind over", "whether changed",
    "whether changed their", "whether changed their mind",
    "whether changed their mind over",
    "whether changed their mind over the",
    "whether changed their mind over the last few",
    "the last few weeks", "actually interesting",
    "angle", "best", "group", "open", "read", "discourse"
]

topic_segment_evidence = topic_segment_evidence[
    ~topic_segment_evidence["dominant_topic"].isin(bad_topics)
]

# Keep only stronger topics
topic_segment_evidence = topic_segment_evidence[
    topic_segment_evidence["engagement_score"] >= 5
]

topic_segment_evidence = topic_segment_evidence.sort_values(
    "engagement_score", ascending=False
)

# Best supporting segment for each topic
best_segment_per_topic = (
    topic_segment_evidence.sort_values("engagement_score", ascending=False)
    .groupby("dominant_topic")
    .head(1)
    .copy()
)

# Simple sponsor category mapping heuristic
sponsor_map = {
    "fitness": "Health & Wellness",
    "health": "Health & Wellness",
    "fatigue": "Health & Wellness",
    "mental health": "Health & Wellness",
    "wellness": "Health & Wellness",
    "career": "Education & Career Services",
    "study": "Education & Career Services",
    "education": "Education & Career Services",
    "work": "Productivity & Career Tools",
    "productivity": "Productivity & Career Tools",
    "money": "Finance & Budgeting",
    "finance": "Finance & Budgeting",
    "travel": "Travel & Lifestyle",
    "lifestyle": "Travel & Lifestyle",
    "fashion": "Beauty & Fashion",
    "beauty": "Beauty & Fashion",
    "food": "Food & Beverage",
    "dating": "Lifestyle & Relationships",
    "relationship": "Lifestyle & Relationships",
    "reality tv": "Entertainment & Streaming",
    "media": "Media & Technology",
    "tech": "Media & Technology"
}

def suggest_sponsor_category(topic):
    topic = str(topic).lower()
    for keyword, category in sponsor_map.items():
        if keyword in topic:
            return category
    return "General Lifestyle / Broad Brand Fit"

# Final sponsor opportunity table
sponsor_opportunity_insights = best_segment_per_topic[[
    "dominant_topic", "city", "industry", "gender", "engagement_score",
    "post_count", "total_likes", "total_comments"
]].copy()

sponsor_opportunity_insights = sponsor_opportunity_insights.rename(columns={
    "dominant_topic": "topic",
    "city": "top_city_segment",
    "industry": "top_industry_segment",
    "gender": "top_gender_segment",
    "engagement_score": "segment_engagement_score"
})

sponsor_opportunity_insights["suggested_sponsor_category"] = (
    sponsor_opportunity_insights["topic"].apply(suggest_sponsor_category)
)

# Reorder columns
sponsor_opportunity_insights = sponsor_opportunity_insights[[
    "topic",
    "suggested_sponsor_category",
    "top_city_segment",
    "top_industry_segment",
    "top_gender_segment",
    "segment_engagement_score",
    "post_count",
    "total_likes",
    "total_comments"
]].sort_values("segment_engagement_score", ascending=False)

save_feature(high_value_demographics, "feature_12_high_value_demographics.csv")
save_feature(topic_segment_evidence, "feature_12_topic_segment_evidence.csv")
save_feature(sponsor_opportunity_insights, "feature_12_sponsor_opportunity_insights.csv")

print("\n=== Feature 12: High Value Demographics ===")
print(high_value_demographics.head(20))

print("\n=== Feature 12: Topic Segment Evidence ===")
print(topic_segment_evidence.head(20))

print("\n=== Feature 12: Sponsor Opportunity Insights ===")
print(sponsor_opportunity_insights.head(20))

Saved: feature_12_high_value_demographics.csv
Saved: feature_12_topic_segment_evidence.csv
Saved: feature_12_sponsor_opportunity_insights.csv

=== Feature 12: High Value Demographics ===
          city     industry             gender  high_value_user_count
26  Gold Coast   Healthcare             female                      3
8     Auckland   Healthcare         non_binary                      2
11    Brisbane   Healthcare             female                      2
40  Manchester      Student             female                      1
45   Melbourne   Healthcare  prefer_not_to_say                      1
44   Melbourne   Government         non_binary                      1
43   Melbourne      Finance         non_binary                      1
42   Melbourne    Education             female                      1
41  Manchester   Technology  prefer_not_to_say                      1
0     Adelaide  Advertising             female                      1
47   Melbourne    Marketing               m

#### AI Implementation 1: Theme Detection

In [117]:
!pip install textblob

#### AI Implementation 2: Sentiment Analysis

In [118]:

def simple_sentiment_score(text):
    text = ai_clean_text(text)
    words = set(text.split())

    positive_words = {
        "love", "loved", "great", "good", "amazing", "fun", "helpful",
        "interesting", "best", "excited", "happy", "strong", "smart",
        "enjoy", "enjoyed", "excellent", "positive", "favourite", "favorite"
    }

    negative_words = {
        "bad", "boring", "hate", "hated", "annoying", "worst", "confusing",
        "poor", "hard", "sad", "frustrating", "negative", "weak", "angry",
        "disappointing", "messy", "problem", "issue"
    }

    positive_count = len(words.intersection(positive_words))
    negative_count = len(words.intersection(negative_words))

    if positive_count == 0 and negative_count == 0:
        return 0.0

    return (positive_count - negative_count) / max(positive_count + negative_count, 1)


try:
    from textblob import TextBlob

    def get_sentiment_score(text):
        text = ai_clean_text(text)
        if not text:
            return 0.0
        return TextBlob(text).sentiment.polarity

except Exception:
    def get_sentiment_score(text):
        return simple_sentiment_score(text)


def get_sentiment_label_from_score(score):
    if score > 0.2:
        return "positive"
    elif score < -0.2:
        return "negative"
    return "neutral"


posts["sentiment_score"] = posts["full_text"].fillna("").apply(get_sentiment_score)
posts["sentiment_label"] = posts["sentiment_score"].apply(get_sentiment_label_from_score)

sentiment_summary = (
    posts.groupby("sentiment_label")
    .agg(
        post_count=("post_id", "count"),
        avg_post_likes=("post_like_count", "mean"),
        avg_comments=("comment_count", "mean"),
        total_post_likes=("post_like_count", "sum"),
        total_comments=("comment_count", "sum")
    )
    .reset_index()
)

sentiment_summary["engagement_score"] = (
    sentiment_summary["avg_post_likes"] + sentiment_summary["avg_comments"]
)

sentiment_summary = sentiment_summary.sort_values(
    "post_count", ascending=False
).reset_index(drop=True)

theme_sentiment_summary = (
    posts.groupby(["theme_label", "sentiment_label"])
    .agg(
        post_count=("post_id", "count"),
        avg_post_likes=("post_like_count", "mean"),
        avg_comments=("comment_count", "mean")
    )
    .reset_index()
    .sort_values(["theme_label", "post_count"], ascending=[True, False])
    .reset_index(drop=True)
)

sentiment_post_evidence = posts[
    [
        "post_id",
        "post_username",
        "post_created_at",
        "heading",
        "theme_label",
        "sentiment_score",
        "sentiment_label",
        "post_like_count",
        "comment_count"
    ]
].copy()

sentiment_post_evidence = sentiment_post_evidence.sort_values(
    ["sentiment_label", "post_created_at"]
).reset_index(drop=True)

save_feature(sentiment_summary, "ai_sentiment_summary.csv")
save_feature(theme_sentiment_summary, "ai_theme_sentiment_summary.csv")
save_feature(sentiment_post_evidence, "ai_sentiment_post_evidence.csv")

print("\n=== AI Implementation 2: Sentiment Summary ===")
print(sentiment_summary)

print("\n=== AI Implementation 2: Theme + Sentiment Summary ===")
print(theme_sentiment_summary.head(15))

print("\n=== AI Implementation 2: Sentiment Evidence ===")
print(sentiment_post_evidence.head(15))

Saved: ai_sentiment_summary.csv
Saved: ai_theme_sentiment_summary.csv
Saved: ai_sentiment_post_evidence.csv

=== AI Implementation 2: Sentiment Summary ===
  sentiment_label  post_count  avg_post_likes  avg_comments  total_post_likes  \
0        positive          47        1.531915      2.021277                72   
1         neutral          36        2.250000      2.611111                81   
2        negative           1        0.000000      2.000000                 0   

   total_comments  engagement_score  
0              95          3.553191  
1              94          4.861111  
2               2          2.000000  

=== AI Implementation 2: Theme + Sentiment Summary ===
                theme_label sentiment_label  post_count  avg_post_likes  \
0                      baby        positive           4        1.750000   
1                      baby         neutral           1        1.000000   
2                     bonus        positive           8        1.500000   
3          

#### AI Implementation 3: Insight Generation

In [121]:
theme_insight_base = (
    posts.groupby("theme_label")
    .agg(
        post_count=("post_id", "count"),
        avg_post_likes=("post_like_count", "mean"),
        avg_comments=("comment_count", "mean"),
        total_post_likes=("post_like_count", "sum"),
        total_comments=("comment_count", "sum")
    )
    .reset_index()
)

theme_insight_base["engagement_score"] = (
    theme_insight_base["avg_post_likes"] + theme_insight_base["avg_comments"]
)

theme_sentiment_counts = (
    posts.pivot_table(
        index="theme_label",
        columns="sentiment_label",
        values="post_id",
        aggfunc="count",
        fill_value=0
    )
    .reset_index()
)

for col in ["positive", "neutral", "negative"]:
    if col not in theme_sentiment_counts.columns:
        theme_sentiment_counts[col] = 0

theme_sentiment_counts["total_posts"] = (
    theme_sentiment_counts["positive"]
    + theme_sentiment_counts["neutral"]
    + theme_sentiment_counts["negative"]
)

theme_sentiment_counts["positive_rate"] = np.where(
    theme_sentiment_counts["total_posts"] > 0,
    theme_sentiment_counts["positive"] / theme_sentiment_counts["total_posts"],
    0
).round(3)

theme_sentiment_counts["negative_rate"] = np.where(
    theme_sentiment_counts["total_posts"] > 0,
    theme_sentiment_counts["negative"] / theme_sentiment_counts["total_posts"],
    0
).round(3)

theme_host_insights = theme_insight_base.merge(
    theme_sentiment_counts[
        ["theme_label", "positive", "neutral", "negative",
         "positive_rate", "negative_rate"]
    ],
    on="theme_label",
    how="left"
)

theme_host_insights = theme_host_insights.sort_values(
    ["post_count", "engagement_score"],
    ascending=[False, False]
).reset_index(drop=True)


def generate_host_takeaway(row):
    theme = row["theme_label"]
    post_count = int(row["post_count"])
    engagement = round(row["engagement_score"], 2)
    pos_rate = row["positive_rate"]
    neg_rate = row["negative_rate"]

    if post_count >= 5 and pos_rate >= 0.5:
        return f"{theme} is a recurring community theme with a mostly positive response. It may be worth exploring further in content or community prompts."

    if post_count >= 5 and neg_rate >= 0.3:
        return f"{theme} is generating repeat discussion, but the tone is more mixed or critical. This may need careful framing."

    if engagement >= 4:
        return f"{theme} is creating stronger engagement than average. It may be a useful area for deeper host attention."

    if post_count <= 2:
        return f"{theme} is currently lower volume. It may be emerging, niche, or not yet a major community focus."

    return f"{theme} is an active discussion area with moderate engagement. It is worth monitoring over time."


theme_host_insights["host_takeaway"] = theme_host_insights.apply(
    generate_host_takeaway,
    axis=1
)

top_discussed = theme_host_insights.sort_values(
    ["post_count", "engagement_score"], ascending=[False, False]
).iloc[0]

top_engagement = theme_host_insights.sort_values(
    "engagement_score", ascending=False
).iloc[0]

most_positive = theme_host_insights.sort_values(
    ["positive_rate", "post_count"], ascending=[False, False]
).iloc[0]

most_critical = theme_host_insights.sort_values(
    ["negative_rate", "post_count"], ascending=[False, False]
).iloc[0]

headline_insights = pd.DataFrame([
    {
        "insight_type": "Most Discussed Theme",
        "theme_label": top_discussed["theme_label"],
        "insight_text": f"The most discussed theme is '{top_discussed['theme_label']}' with {int(top_discussed['post_count'])} posts."
    },
    {
        "insight_type": "Highest Engagement Theme",
        "theme_label": top_engagement["theme_label"],
        "insight_text": f"The highest-engagement theme is '{top_engagement['theme_label']}' with an engagement score of {round(top_engagement['engagement_score'], 2)}."
    },
    {
        "insight_type": "Most Positive Theme",
        "theme_label": most_positive["theme_label"],
        "insight_text": f"The most positively received theme is '{most_positive['theme_label']}', with a positive rate of {round(most_positive['positive_rate'] * 100, 1)}%."
    },
    {
        "insight_type": "Most Critical Theme",
        "theme_label": most_critical["theme_label"],
        "insight_text": f"The most critical or mixed-response theme is '{most_critical['theme_label']}', with a negative rate of {round(most_critical['negative_rate'] * 100, 1)}%."
    }
])

insight_evidence = posts[
    [
        "post_id",
        "post_username",
        "post_created_at",
        "heading",
        "theme_label",
        "sentiment_label",
        "sentiment_score",
        "post_like_count",
        "comment_count"
    ]
].copy()

save_feature(theme_host_insights, "ai_host_theme_insights.csv")
save_feature(headline_insights, "ai_headline_insights.csv")
save_feature(insight_evidence, "ai_insight_evidence.csv")

print("\n=== AI Implementation 3: Headline Insights ===")
print(headline_insights)

print("\n=== AI Implementation 3: Theme Host Insights ===")
print(theme_host_insights.head(15))

print("\n=== AI Implementation 3: Evidence Sample ===")
print(insight_evidence.head(15))

Saved: ai_host_theme_insights.csv
Saved: ai_headline_insights.csv
Saved: ai_insight_evidence.csv

=== AI Implementation 3: Headline Insights ===
               insight_type       theme_label  \
0      Most Discussed Theme  hearing versions   
1  Highest Engagement Theme           curious   
2       Most Positive Theme             bonus   
3       Most Critical Theme              dumb   

                                        insight_text  
0  The most discussed theme is 'hearing versions'...  
1  The highest-engagement theme is 'curious' with...  
2  The most positively received theme is 'bonus',...  
3  The most critical or mixed-response theme is '...  

=== AI Implementation 3: Theme Host Insights ===
                 theme_label  post_count  avg_post_likes  avg_comments  \
0           hearing versions          14        1.571429      2.571429   
1                       dumb          11        2.000000      2.545455   
2  separate interesting loud          11        2.000000      

In [123]:
theme_host_insights[["theme_label", "post_count", "engagement_score", "positive_rate", "negative_rate", "host_takeaway"]].head(10)

,theme_label,post_count,engagement_score,positive_rate,negative_rate,host_takeaway
0,hearing versions,14,4.142857,0.357,0.000,hearing versions is creating stronger engageme...
1,dumb,11,4.545455,0.455,0.091,dumb is creating stronger engagement than aver...
2,separate interesting loud,11,3.818182,0.818,0.000,separate interesting loud is a recurring commu...
3,makes,10,4.300000,0.200,0.000,makes is creating stronger engagement than ave...
4,feels interesting angle,9,4.333333,0.667,0.000,feels interesting angle is a recurring communi...
5,rattling head latest,9,3.222222,0.889,0.000,rattling head latest is a recurring community ...
6,bonus,8,3.125000,1.000,0.000,bonus is a recurring community theme with a mo...
7,curious,7,5.571429,0.000,0.000,curious is creating stronger engagement than a...
8,baby,5,3.800000,0.800,0.000,baby is a recurring community theme with a mos...
